In [ ]:
import os
import sys
import tempfile

PROJECT_DIR = r"/Users/jiezhao/Documents/Harvard Data Sciense/CSCI-104/week8_LLM_LangChain/project_chat_box"
os.chdir(PROJECT_DIR)

if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from loadllm import Loadllm

pdf_path = "your_test_file.pdf"   # change this

loader = PyMuPDFLoader(pdf_path)
docs = loader.load()

print("pages loaded:", len(docs))
print(docs[0].page_content[:500])

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = FAISS.from_documents(docs, embeddings)

llm = Loadllm.load_llm()

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the context below to answer the question.\n\nContext: {context}"),
    ("human", "{input}")
])

combine_docs_chain = create_stuff_documents_chain(llm, prompt)
chain = create_retrieval_chain(db.as_retriever(), combine_docs_chain)